In [1]:
# -*- coding: utf-8 -*-
"""
Capítulo 1: Fundamentos de Python aplicado a datos
Sección 2: Pandas y mini ETL
Bloque 1: Introducción a Pandas y DataFrames

Este script cubre los siguientes subtemas (ST):
ST1: Introducción a Pandas como herramienta de manipulación de datos.
ST2: Limitaciones de listas/diccionarios vs DataFrames.
ST3: Concepto de Series y DataFrame.
ST4: Creación de DataFrames desde estructuras nativas.
ST5: Lectura de datos con Pandas (read_csv).
ST6: Inspección inicial (head, shape, info).
ST7: Tipos de columnas en Pandas.
ST8: Exploración básica de datos.
ST9: Contexto de uso en pipelines.
ST10: Escalabilidad conceptual (limitaciones de Pandas vs Big Data).
"""

import pandas as pd
import random
import os

# Fijar semilla para operaciones aleatorias si se requieren
random.seed(987654)

# =============================================================================
# ST1, ST2, ST3, ST4: Creación de Series y DataFrames desde estructuras nativas
# =============================================================================
print("--- ST1 a ST4: Creación de Series y DataFrames ---")

# Mostrar limitación de listas nativas (operaciones matemáticas requieren bucles)
precios_lista = [10.5, 20.0, 15.75, 40.2]
# Intentar multiplicar precios_lista * 2 duplicaría la lista, no los valores.
# Con Pandas (Series), las operaciones son vectorizadas.

# Crear una Serie en Pandas (Unidimensional)
precios_serie = pd.Series(precios_lista, name="Precio")
print("Serie de Pandas:\n", precios_serie)
print("\nMultiplicar Serie por 2 (Vectorizado):\n", precios_serie * 2)

# Crear un DataFrame desde un diccionario (Bidimensional)
datos_diccionario = {
    "id_transaccion": [1, 2, 3, 4],
    "cliente": ["Ana", "Luis", "Marta", "Pedro"],
    "monto": [150.50, 20.00, 340.10, 99.99],
    "categoria": ["Electrónica", "Ropa", "Electrónica", "Hogar"]
}

df_nombres = pd.DataFrame(datos_diccionario)
print("\nDataFrame creado desde un diccionario:\n", df_nombres)

--- ST1 a ST4: Creación de Series y DataFrames ---
Serie de Pandas:
 0    10.50
1    20.00
2    15.75
3    40.20
Name: Precio, dtype: float64

Multiplicar Serie por 2 (Vectorizado):
 0    21.0
1    40.0
2    31.5
3    80.4
Name: Precio, dtype: float64

DataFrame creado desde un diccionario:
    id_transaccion cliente   monto    categoria
0               1     Ana  150.50  Electrónica
1               2    Luis   20.00         Ropa
2               3   Marta  340.10  Electrónica
3               4   Pedro   99.99        Hogar


In [2]:
archivo_csv = "transacciones_b1.csv"
transacciones_csv = "id_trx,fecha,id_cliente,monto,estado,metodo_pago\n"
estados = ["Completada", "Pendiente", "Fallida"]
metodos = ["Tarjeta", "Transferencia", "Efectivo"]
for i in range(1, 101):
    fecha = f"2023-10-{random.randint(1, 31):02d}"
    id_cliente = random.randint(1000, 1050)
    monto = round(random.uniform(10.0, 500.0), 2)
    estado = random.choice(estados)
    metodo = random.choice(metodos)
    # Introducir algunos nulos aleatorios en monto para realismo
    monto_str = str(monto) if random.random() > 0.05 else ""
    transacciones_csv += f"{i},{fecha},{id_cliente},{monto_str},{estado},{metodo}\n"

In [3]:
with open(archivo_csv, "w", encoding="utf-8") as f:
    f.write(transacciones_csv)

In [4]:
print(f"\n[INFO] Archivo {archivo_csv} generado con éxito.")


[INFO] Archivo transacciones_b1.csv generado con éxito.


In [5]:
# =============================================================================
# ST5: Lectura de datos con Pandas (read_csv)
# =============================================================================
print("\n--- ST5: Lectura de datos con Pandas ---")
# Leer el archivo CSV recién generado
# En un pipeline, este es el paso de "Ingesta" o "Extracción"
df_transacciones = pd.read_csv(archivo_csv)
print("Archivo CSV leído y cargado en el DataFrame 'df_transacciones'.")


--- ST5: Lectura de datos con Pandas ---
Archivo CSV leído y cargado en el DataFrame 'df_transacciones'.


In [6]:
# =============================================================================
# ST6: Inspección inicial (head, shape, info)
# =============================================================================
print("\n--- ST6: Inspección inicial del DataFrame ---")

# Mostrar las primeras 5 filas (head)
print("\nPrimeras 5 filas (df.head()):")
print(df_transacciones.head())


--- ST6: Inspección inicial del DataFrame ---

Primeras 5 filas (df.head()):
   id_trx       fecha  id_cliente   monto      estado metodo_pago
0       1  2023-10-04        1035  200.68     Fallida    Efectivo
1       2  2023-10-11        1007   10.74     Fallida    Efectivo
2       3  2023-10-06        1001  220.31  Completada    Efectivo
3       4  2023-10-15        1008   49.55   Pendiente     Tarjeta
4       5  2023-10-15        1036     NaN   Pendiente    Efectivo


In [7]:
# Mostrar la dimensión del DataFrame (filas, columnas)
print("\nDimensiones del dataset (df.shape):", df_transacciones.shape)

# Mostrar información general del DataFrame (tipos de datos, nulos, memoria)
print("\nInformación del DataFrame (df.info()):")
df_transacciones.info()


Dimensiones del dataset (df.shape): (100, 6)

Información del DataFrame (df.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_trx       100 non-null    int64  
 1   fecha        100 non-null    object 
 2   id_cliente   100 non-null    int64  
 3   monto        95 non-null     float64
 4   estado       100 non-null    object 
 5   metodo_pago  100 non-null    object 
dtypes: float64(1), int64(2), object(3)
memory usage: 4.8+ KB


In [8]:
print(df_transacciones.head())

   id_trx       fecha  id_cliente   monto      estado metodo_pago
0       1  2023-10-04        1035  200.68     Fallida    Efectivo
1       2  2023-10-11        1007   10.74     Fallida    Efectivo
2       3  2023-10-06        1001  220.31  Completada    Efectivo
3       4  2023-10-15        1008   49.55   Pendiente     Tarjeta
4       5  2023-10-15        1036     NaN   Pendiente    Efectivo


In [10]:
df_transacciones.shape

(100, 6)

In [11]:
df_transacciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_trx       100 non-null    int64  
 1   fecha        100 non-null    object 
 2   id_cliente   100 non-null    int64  
 3   monto        95 non-null     float64
 4   estado       100 non-null    object 
 5   metodo_pago  100 non-null    object 
dtypes: float64(1), int64(2), object(3)
memory usage: 4.8+ KB


In [12]:
df_transacciones.dtypes

id_trx           int64
fecha           object
id_cliente       int64
monto          float64
estado          object
metodo_pago     object
dtype: object

In [14]:
df_transacciones.describe()

,id_trx,id_cliente,monto
count,100.000000,100.000000,95.000000
mean,50.500000,1024.160000,231.711789
std,29.011492,15.448003,141.817041
min,1.000000,1000.000000,10.740000
25%,25.750000,1009.000000,109.370000
50%,50.500000,1024.500000,220.310000
75%,75.250000,1037.000000,334.655000
max,100.000000,1050.000000,495.950000


In [15]:
df_transacciones['estado'].value_counts()

estado
Pendiente     39
Fallida       31
Completada    30
Name: count, dtype: int64

In [17]:
df_transacciones['metodo_pago'].value_counts()

metodo_pago
Tarjeta          39
Efectivo         34
Transferencia    27
Name: count, dtype: int64

In [18]:
df_transacciones.describe

<bound method NDFrame.describe of     id_trx       fecha  id_cliente   monto      estado    metodo_pago
0        1  2023-10-04        1035  200.68     Fallida       Efectivo
1        2  2023-10-11        1007   10.74     Fallida       Efectivo
2        3  2023-10-06        1001  220.31  Completada       Efectivo
3        4  2023-10-15        1008   49.55   Pendiente        Tarjeta
4        5  2023-10-15        1036     NaN   Pendiente       Efectivo
..     ...         ...         ...     ...         ...            ...
95      96  2023-10-18        1049  102.61     Fallida       Efectivo
96      97  2023-10-14        1018  331.75  Completada        Tarjeta
97      98  2023-10-08        1004   23.39   Pendiente        Tarjeta
98      99  2023-10-12        1001  138.24   Pendiente        Tarjeta
99     100  2023-10-06        1001  314.59   Pendiente  Transferencia

[100 rows x 6 columns]>

In [19]:
df_transacciones['fecha'].value_counts()

fecha
2023-10-11    6
2023-10-28    6
2023-10-04    5
2023-10-12    5
2023-10-05    4
2023-10-10    4
2023-10-03    4
2023-10-27    4
2023-10-01    4
2023-10-21    4
2023-10-22    4
2023-10-15    4
2023-10-20    4
2023-10-29    3
2023-10-18    3
2023-10-16    3
2023-10-13    3
2023-10-30    3
2023-10-08    3
2023-10-07    3
2023-10-06    3
2023-10-26    2
2023-10-24    2
2023-10-02    2
2023-10-23    2
2023-10-17    2
2023-10-09    2
2023-10-14    2
2023-10-25    2
2023-10-19    1
2023-10-31    1
Name: count, dtype: int64